In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score

In [ ]:
# ------------------
# Reproducibility
# ------------------

def set_seed(seed: int):
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True,warn_only=True)

seed = 50
set_seed(seed)

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------
# Load dataset
# ---------------

df = pandas.read_csv("/content/drive/MyDrive/data/dataset.csv")

if "hash" in df.columns:
  df = df.drop(columns=["hash"])

# Balanced Dataset:
# df_majority = df[df["malware"] == 1]
# df_minority = df[df["malware"] == 0]

# df_majority_down = df_majority.sample(n=len(df_minority), random_state=42)
# df_balanced = pandas.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

# df = df_balanced

In [ ]:
X = df.drop(columns=['malware']).values.astype(numpy.float32)
y = df['malware'].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df['malware'], random_state=42
)

In [ ]:
X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)
y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

# class_sample_counts = numpy.bincount(y_train.cpu().numpy())
# class_weights = 1. / class_sample_counts
# class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
    # shuffle=True
)

In [ ]:
# Hyperparameters
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
EMB_DIM = 128
EPOCHS = 100

In [ ]:
# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)

        self.conv = nn.Sequential(
            nn.Conv1d(EMB_DIM, 128, 5, padding=2),
            nn.LeakyReLU(0.2),
            nn.Conv1d(128, 256, 5, padding=2),
            nn.LeakyReLU(0.2),
            nn.AdaptiveMaxPool1d(1)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = self.fc(x)
        return x


D = Discriminator().to(DEVICE)
optimizer = optim.Adam(D.parameters(), lr=2e-4)

In [70]:
# Training
history = {
    "loss": []
}

for epoch in range(EPOCHS):
    D.train()
    total_loss = 0

    for real_x, real_y in train_loader:

        optimizer.zero_grad()

        logits = D(real_x)

        loss = F.cross_entropy(
            logits,
            real_y,
            # weight=class_weights
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")
    history['loss'].append(total_loss/len(train_loader))

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 1 | Loss: 0.0994
Epoch 2 | Loss: 0.0371
Epoch 3 | Loss: 0.0202
Epoch 4 | Loss: 0.0134
Epoch 5 | Loss: 0.0104
Epoch 6 | Loss: 0.0098
Epoch 7 | Loss: 0.0091
Epoch 8 | Loss: 0.0083
Epoch 9 | Loss: 0.0080
Epoch 10 | Loss: 0.0066
Epoch 11 | Loss: 0.0059
Epoch 12 | Loss: 0.0059
Epoch 13 | Loss: 0.0052
Epoch 14 | Loss: 0.0052
Epoch 15 | Loss: 0.0048
Epoch 16 | Loss: 0.0045
Epoch 17 | Loss: 0.0047
Epoch 18 | Loss: 0.0047
Epoch 19 | Loss: 0.0046
Epoch 20 | Loss: 0.0040
Epoch 21 | Loss: 0.0037
Epoch 22 | Loss: 0.0040
Epoch 23 | Loss: 0.0039
Epoch 24 | Loss: 0.0042
Epoch 25 | Loss: 0.0039
Epoch 26 | Loss: 0.0043
Epoch 27 | Loss: 0.0046
Epoch 28 | Loss: 0.0043
Epoch 29 | Loss: 0.0039
Epoch 30 | Loss: 0.0034
Epoch 31 | Loss: 0.0036
Epoch 32 | Loss: 0.0033
Epoch 33 | Loss: 0.0033
Epoch 34 | Loss: 0.0035
Epoch 35 | Loss: 0.0040
Epoch 36 | Loss: 0.0035
Epoch 37 | Loss: 0.0033
Epoch 38 | Loss: 0.0033
Epoch 39 | Loss: 0.0034
Epoch 40 | Loss: 0.0035
Epoch 41 | Loss: 0.0045
Epoch 42 | Loss: 0.0038
E

In [71]:
history_df = pandas.DataFrame(history)
history_df.to_csv(f"cnn_history_{seed}.csv")
torch.save(D.state_dict(), f"cnn_seed_{seed}.pt")

In [72]:
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(
    y_val.cpu().numpy(),
    preds.cpu().numpy(),
    digits=4
))

print("PR-AUC:",
      average_precision_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

print("ROC-AUC:",
      roc_auc_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

              precision    recall  f1-score   support

           0     0.9293    0.8519    0.8889       324
           1     0.9963    0.9984    0.9973     12839

    accuracy                         0.9948     13163
   macro avg     0.9628    0.9251    0.9431     13163
weighted avg     0.9946    0.9948    0.9946     13163

PR-AUC: 0.999876903235778
ROC-AUC: 0.9953237339164333


### Results Compilation
-----

**Seed 10**
```
         precision    recall  f1-score   support

           0     0.9392    0.8580    0.8968       324
           1     0.9964    0.9986    0.9975     12839

    accuracy                         0.9951     13163
   macro avg     0.9678    0.9283    0.9471     13163
weighted avg     0.9950    0.9951    0.9950     13163

PR-AUC: 0.9998551171994043
ROC-AUC: 0.9947015940051485
```

**Seed 20**
```
              precision    recall  f1-score   support

           0     0.9723    0.8673    0.9168       324
           1     0.9967    0.9994    0.9980     12839

    accuracy                         0.9961     13163
   macro avg     0.9845    0.9333    0.9574     13163
weighted avg     0.9961    0.9961    0.9960     13163

PR-AUC: 0.9998734888729256
ROC-AUC: 0.9952226482005541
```

**Seed 30**
```
              precision    recall  f1-score   support

           0     0.9213    0.8673    0.8935       324
           1     0.9967    0.9981    0.9974     12839

    accuracy                         0.9949     13163
   macro avg     0.9590    0.9327    0.9454     13163
weighted avg     0.9948    0.9949    0.9948     13163

PR-AUC: 0.9997948333304536
ROC-AUC: 0.9940885890693769
```

**Seed 40**
```
              precision    recall  f1-score   support

           0     0.9293    0.8519    0.8889       324
           1     0.9963    0.9984    0.9973     12839

    accuracy                         0.9948     13163
   macro avg     0.9628    0.9251    0.9431     13163
weighted avg     0.9946    0.9948    0.9946     13163

PR-AUC: 0.999876903235778
ROC-AUC: 0.9953237339164333
```

**Seed 50**
```
              precision    recall  f1-score   support

           0     0.9813    0.8117    0.8885       324
           1     0.9953    0.9996    0.9974     12839

    accuracy                         0.9950     13163
   macro avg     0.9883    0.9057    0.9430     13163
weighted avg     0.9949    0.9950    0.9948     13163

PR-AUC: 0.9996224535717663
ROC-AUC: 0.9896024747129455
```